<a href="https://colab.research.google.com/github/ehsanre1376/YouTube-DownLoader-To-Colab/blob/main/YouTubeToGoogleDrive3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Installation and Setup ---
!pip install yt-dlp ipywidgets -q
# Use apt-get update before install to ensure package lists are fresh
!sudo apt-get update -qq && sudo apt-get install ffmpeg -qq

import os
import re
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from yt_dlp import YoutubeDL
from google.colab import drive
import logging
import time # For potential delays if needed

# --- Configuration ---
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

# Mount Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
    BASE_DRIVE_PATH = '/content/drive/MyDrive/'
    drive_mounted = True
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
    print("Downloads will be saved to Colab's temporary storage (/content/downloads) instead.")
    BASE_DRIVE_PATH = '/content/downloads/' # Fallback path
    drive_mounted = False
    os.makedirs(BASE_DRIVE_PATH, exist_ok=True)

# --- Helper Functions ---
def sanitize_name(name):
    """Removes characters invalid for Windows/Linux filenames."""
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = re.sub(r'\.{2,}', '.', name)
    name = name.strip(' .')
    if not name:
        name = "untitled"
    return name

def get_base_output_directory(info, output_base, media_type):
    """Determines the primary output directory based on media type and info."""
    title = "Unknown"
    if media_type == 'playlist':
        title = info.get('title') or info.get('playlist_title', 'Unknown Playlist')
    else: # video
        title = info.get('title', 'Unknown Video')

    sanitized_title = sanitize_name(title)
    return os.path.join(output_base, sanitized_title)


# --- Main Download Logic ---
def download_media(url, media_type, output_base, progress_callback, initial_info):
    """Downloads media using yt-dlp with specified options."""

    # Determine the final base directory using the initial info
    # This ensures the folder name matches the overall playlist/video title
    final_base_dir = get_base_output_directory(initial_info, output_base, media_type)
    os.makedirs(final_base_dir, exist_ok=True) # Create the final directory structure

    # --- Determine Output Template ---
    if media_type == 'playlist':
         # Playlist: Save under Playlist Title / Video Title / video.mp4
         # yt-dlp automatically uses %(playlist_title)s if available for the base path,
         # but we use the explicitly created final_base_dir for consistency.
         # Individual videos within the playlist get their own subfolder.
         template = os.path.join(final_base_dir, '%(title)s', '%(title)s.%(ext)s')
    else:
         # Single Video: Save under Video Title / video.mp4
         template = os.path.join(final_base_dir, '%(title)s.%(ext)s')

    # Define yt-dlp options
    ydl_opts = {
        'format': 'bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best[height<=1080][ext=mp4]/best[height<=1080]',
        'merge_output_format': 'mp4',
        'writesubtitles': True,
        'subtitleslangs': ['en', 'fa'],
        'writeautomaticsub': True,
        'subtitlesformat': 'srt',
        'quiet': True,
        'no_warnings': True,
        'progress_hooks': [progress_callback],
        'postprocessors': [{
            'key': 'FFmpegVideoConvertor',
            'preferedformat': 'mp4',
        }],
        'postprocessor_args': {
             'ffmpeg': ['-loglevel', 'error']
        },
        'fragment_retries': 10,
        'retry_sleep_functions': {'http': lambda n: min(n * 5, 30),
                                  'fragment': lambda n: min(n * 5, 30)},
        'ignoreerrors': True, # Important for playlists
        'outtmpl': template, # Use the determined template
        # 'writethumbnail': True, # Optionally download thumbnail
    }

    # --- Execute Download ---
    download_error = None
    try:
        with YoutubeDL(ydl_opts) as ydl:
            status_widget.value = f"<i>Starting download ({media_type})...</i>"
            # We already extracted info, just download now
            ydl.download([url])
    except Exception as e:
        logger.error(f"yt-dlp download failed: {e}", exc_info=True)
        download_error = str(e)

    return final_base_dir, download_error


# --- GUI Components ---
url_widget = widgets.Text(
    placeholder='Enter YouTube Video or Playlist URL',
    layout={'width': '95%'}
)
# REMOVED type_widget
drive_folder_widget = widgets.Text(
    value='YT_Downloads',
    description='Drive Folder:',
    placeholder='Subfolder in MyDrive (e.g., Videos/Tutorials)',
     layout={'width': '95%'}
)
download_btn = widgets.Button(
    description='Start Download',
    button_style='success',
    icon='download'
)
progress_widget = widgets.FloatProgress(
    value=0.0,
    min=0.0,
    max=1.0,
    description='Progress:',
    bar_style='info',
    orientation='horizontal',
    layout={'width': '95%'}
)
status_widget = widgets.HTML(
    value="<b>Status:</b> Ready"
)
output_widget = widgets.Output()

# --- Button Click Handler ---
def on_button_click(b):
    # Disable button immediately
    download_btn.disabled = True
    download_btn.description = 'Processing...'
    download_btn.button_style = 'warning'
    progress_widget.value = 0.0
    progress_widget.bar_style = 'info'
    status_widget.value = "<b>Status:</b> Initializing..."

    with output_widget:
        clear_output(wait=True) # Clear previous logs

    url = url_widget.value.strip()
    drive_subfolder = drive_folder_widget.value.strip()

    if not url:
        status_widget.value = "<b style='color:red;'>Error:</b> Please enter a URL."
        download_btn.disabled = False
        download_btn.description = 'Start Download'
        download_btn.button_style = 'success'
        return

    # Construct the base output path
    sanitized_subfolder = sanitize_name(drive_subfolder) if drive_subfolder else "YT_Downloads"
    full_output_base_path = os.path.join(BASE_DRIVE_PATH, sanitized_subfolder)

    # --- Progress Hook (defined inside to access widgets directly) ---
    last_filename = ""

    def progress_hook(d):
        nonlocal last_filename
        try:
            if d['status'] == 'downloading':
                percent_str = d.get('_percent_str', '0%').strip().strip('%')
                try:
                    # *** FIX: Convert percentage string to float ***
                    progress_widget.value = float(percent_str) / 100.0
                except ValueError:
                    logger.warning(f"Could not parse percentage: {percent_str}")
                    # Optionally set to 0 or leave as is
                    # progress_widget.value = 0.0

                filename = d.get('filename', 'Unknown file')
                if filename != last_filename:
                     last_filename = filename
                     base_filename = os.path.basename(filename)
                     display_filename = (base_filename[:50] + '...') if len(base_filename) > 53 else base_filename
                     status_widget.value = f"<i>Downloading: {display_filename} ({d.get('_speed_str', '?')})</i>"
                progress_widget.bar_style = 'info'

            elif d['status'] == 'error':
                 logger.error(f"Error reported by yt-dlp hook for: {d.get('filename')}")
                 # Don't make the whole status red, just log it. ignoreerrors is on.
                 # status_widget.value = f"<b style='color:orange;'>Warning:</b> Error processing {os.path.basename(d.get('filename', 'a file'))}."
                 progress_widget.bar_style = 'warning'

            elif d['status'] == 'finished':
                # Only set to 100% if it's the final file processing, might be tricky with playlists
                # Let's rely on the final status update after download completes.
                # progress_widget.value = 1.0
                base_filename = os.path.basename(d.get('filename', 'a file'))
                display_filename = (base_filename[:50] + '...') if len(base_filename) > 53 else base_filename
                status_widget.value = f"<i>Finished: {display_filename}. Post-processing...</i>"
                progress_widget.bar_style = 'info' # Back to info for post-processing

            elif d['status'] == 'postprocessing':
                 status_widget.value = f"<i>Post-processing: {d.get('postprocessor', 'ffmpeg')}...</i>"

        except Exception as e:
            logger.error(f"Error in progress_hook: {e}", exc_info=True)
            # Avoid crashing the hook itself
            status_widget.value = "<b style='color:orange;'>Warning:</b> Internal error in progress update."


    # --- Auto-detect Type and Execute Download ---
    final_path = None
    download_error_msg = None
    media_type = 'video' # Default assumption
    initial_info = None

    try:
        # 1. Extract basic info to detect type
        status_widget.value = "<i>Detecting URL type...</i>"
        # extract_flat is faster for playlists, simulate=True avoids some processing
        info_opts = {'quiet': True, 'no_warnings': True, 'extract_flat': True, 'simulate': True}
        with YoutubeDL(info_opts) as ydl_info:
             initial_info = ydl_info.extract_info(url, download=False)

        # 2. Determine type
        if initial_info and initial_info.get('entries') and isinstance(initial_info.get('entries'), list):
            media_type = 'playlist'
            status_widget.value = f"<i>Playlist detected: '{sanitize_name(initial_info.get('title', '...'))}'</i>"
        else:
            media_type = 'video'
            status_widget.value = f"<i>Single video detected: '{sanitize_name(initial_info.get('title', '...'))}'</i>"

        time.sleep(0.5) # Small delay for user to see the detected type message

        # 3. Start the actual download
        if initial_info: # Proceed only if info extraction was successful
            final_path, download_error_msg = download_media(
                url,
                media_type,
                full_output_base_path,
                progress_hook,
                initial_info # Pass the fetched info
            )
        else:
             download_error_msg = "Failed to extract initial video/playlist information."


        # 4. Report final status
        if download_error_msg:
             # Check if it was just an 'ignoreerrors' case within a playlist
             if final_path and media_type == 'playlist':
                 status_widget.value = (f"<b style='color:orange;'>Warning:</b> Some items in playlist might have failed. "
                                        f"Content saved to: <a href='file://{final_path}' target='_blank'>{final_path}</a>")
                 progress_widget.bar_style = 'warning'
             else:
                 status_widget.value = f"<b style='color:red;'>Error:</b> {download_error_msg}"
                 progress_widget.bar_style = 'danger'
        elif final_path:
             progress_widget.value = 1.0
             progress_widget.bar_style = 'success'
             if drive_mounted:
                 # Creating a clickable link to Drive folder in Colab output is tricky/unreliable
                 # Just show the path clearly. User can navigate in the Drive UI.
                 status_widget.value = (f"<b>Success!</b> Content saved to: "
                                        f"<code>{final_path}</code>")
             else:
                 status_widget.value = f"<b>Success!</b> Content saved to: <code>{final_path}</code>"
        else:
             status_widget.value = "<b style='color:red;'>Error:</b> Download finished but final path is unknown."
             progress_widget.bar_style = 'danger'

    except Exception as e:
        logger.error(f"An critical error occurred: {e}", exc_info=True)
        status_widget.value = f"<b style='color:red;'>Critical Error:</b> {str(e)}"
        progress_widget.bar_style = 'danger'
        # import traceback # Uncomment for debugging if needed
        # with output_widget:
        #    traceback.print_exc()

    finally:
        # Re-enable button
        download_btn.disabled = False
        download_btn.description = 'Start Download'
        if progress_widget.bar_style == 'success':
             download_btn.button_style = 'success'
        elif progress_widget.bar_style == 'warning':
             download_btn.button_style = 'warning'
        else: # danger or info (if stopped early)
             download_btn.button_style = 'danger'


# --- Display GUI ---
download_btn.on_click(on_button_click)

drive_status_html = ""
if not drive_mounted:
    drive_status_html = "<p style='color: orange; font-weight: bold;'>Warning: Google Drive not mounted. Files will be saved to temporary Colab storage and may be lost when the runtime disconnects.</p>"


gui = widgets.VBox([
    widgets.HTML(value=drive_status_html),
    widgets.HTML("<h2>YouTube Downloader</h2>"),
    widgets.Label("Enter the URL of the YouTube video or playlist:"),
    url_widget,
    # Removed the HBox with type_widget, only folder widget remains
    drive_folder_widget,
    download_btn,
    widgets.HTML("<hr>"),
    progress_widget,
    status_widget,
    output_widget
])

display(gui)